# Ceftazidime x *E. coli*: LR, MLP, Random Forest

**Cross-site** (train on A) and **Aggregated** (train on pooled A+B+C+D).

Both use **85/15** stratified splits. All hyperparameter and threshold tuning happens within the 85% train (CV / sub-split) -- the 15% holdout is only evaluated once.

**Models:** Logistic Regression (L2, class_weight="balanced"), MLP (6x6 grid lr x dropout), Random Forest (GridSearchCV)

**Output:** Two heatmaps -- Balanced Accuracy and AUC-ROC -- Methods x Sites

Compatible with Google Colab.

In [ ]:
!pip install maldideepkit maldiamrkit seaborn --quiet

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted')
    IN_COLAB = True
except ImportError:
    print('Running locally')
    IN_COLAB = False

In [ ]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (GridSearchCV, StratifiedKFold,
                                      train_test_split, cross_val_predict)
from sklearn.metrics import (balanced_accuracy_score, roc_auc_score)

from maldideepkit.attention.mlp import MaldiMLPClassifier
from maldideepkit.base.data import fit_input_transform, apply_input_transform

warnings.filterwarnings("ignore", category=FutureWarning)
SEED = 42
np.random.seed(SEED)
SEED_TUNE = 123  # separate seed for internal tuning splits

In [ ]:
if IN_COLAB:
    DRYAD = Path("/content/drive/MyDrive/Flower/DRIAMS-DataSet")
else:
    DRYAD = Path("/media/asd/8f69beed-e984-445f-b8b3-abbb6a1a4b3f/Dryad-DataSet")

OUT_DIR = Path("./results_ceftazidime_ecoli")
OUT_DIR.mkdir(exist_ok=True)

SITES_PATHS = {
    "A": DRYAD / "Processed/Proc_DRIAMS-A" / "Ceftazidime" / "data.csv",
    "B": DRYAD / "Processed/Proc_DRIAMS-B" / "Ceftazidime" / "data.csv",
    "C": DRYAD / "Processed/Proc_DRIAMS-C" / "Ceftazidime" / "data.csv",
    "D": DRYAD / "Processed/Proc_DRIAMS-D" / "Ceftazidime" / "data.csv",
}
print(f"Output dir: {OUT_DIR.resolve()}")

In [ ]:
# ── Load Ceftazidime + E. coli from all 4 sites ──
SPECIES = "Escherichia coli"
site_data = {}

for site, path in SITES_PATHS.items():
    if not path.exists():
        print(f"  WARNING: {path} does not exist, skipping site {site}")
        continue
    df = pd.read_csv(path)
    df_eco = df[df["species"] == SPECIES].copy()
    bin_cols = [c for c in df_eco.columns if c.startswith("bin_")]
    X = df_eco[bin_cols].to_numpy(dtype="float32")
    y = df_eco["label"].to_numpy(dtype="int64")
    site_data[site] = {"X": X, "y": y, "n": len(y)}
    n_r = (y == 1).sum()
    n_s = (y == 0).sum()
    print(f"  Site {site}: {len(y)} samples ({n_s} S, {n_r} R, {n_r/len(y)*100:.1f}% R)")

print(f"\nLoaded {len(site_data)} sites.")

In [ ]:
# ── Cross-site split: A -> 85/15 (train / holdout) ──
X_A, y_A = site_data["A"]["X"], site_data["A"]["y"]
X_A_train, X_A_val, y_A_train, y_A_val = train_test_split(
    X_A, y_A, test_size=0.15, stratify=y_A, random_state=SEED)
print(f"Cross-site: A train={len(X_A_train)}  A val={len(X_A_val)}  "
      f"(holdout: {len(X_A_val)/len(X_A)*100:.1f}%)")

In [ ]:
# ── Aggregated split: A+B+C+D -> 85/15 ──
available = [s for s in "ABCD" if s in site_data]
X_all = np.concatenate([site_data[s]["X"] for s in available])
y_all = np.concatenate([site_data[s]["y"] for s in available])
sp_all = np.concatenate([np.full(site_data[s]["n"], s) for s in available])

X_agg_train, X_agg_val, y_agg_train, y_agg_val = train_test_split(
    X_all, y_all, test_size=0.15, stratify=y_all, random_state=SEED)
print(f"Aggregated: train={len(X_agg_train)}  val={len(X_agg_val)}  "
      f"(holdout: {len(X_agg_val)/len(X_all)*100:.1f}%)")

---
## Preprocessing

In [ ]:
# ── Preprocessing: log1p + standardise (fit on train, apply to all) ──

# Cross-site: fit on A-train
state_cs = fit_input_transform(X_A_train, "log1p+standardize")
X_A_train_pp = apply_input_transform(X_A_train, state_cs)
X_A_val_pp   = apply_input_transform(X_A_val, state_cs)

# Test sites B / C / D
test_pp = {}
for site in "BCD":
    if site not in site_data:
        continue
    X_t = apply_input_transform(site_data[site]["X"], state_cs)
    test_pp[site] = (X_t, site_data[site]["y"])
    print(f"  Test {site}: {len(X_t)} samples preprocessed")

# B+C+D combined test
if test_pp:
    X_BCD = np.concatenate([test_pp[s][0] for s in "BCD" if s in test_pp])
    y_BCD = np.concatenate([test_pp[s][1] for s in "BCD" if s in test_pp])
    test_pp["B+C+D"] = (X_BCD, y_BCD)
    print(f"  Test B+C+D: {len(X_BCD)} samples")

# Aggregated: fit on agg-train
state_agg = fit_input_transform(X_agg_train, "log1p+standardize")
X_agg_train_pp = apply_input_transform(X_agg_train, state_agg)
X_agg_val_pp   = apply_input_transform(X_agg_val, state_agg)
print("\nPreprocessing done.")

In [ ]:
# ── Shared constants for tuning ──
C_GRID = np.linspace(5e-5, 1e-3, 15)
LR_GRID = np.linspace(1e-4, 5e-4, 6)
DROP_GRID = np.linspace(0.2, 0.6, 6)
THRESHOLDS = np.linspace(0.05, 0.95, 91)

RF_PARAM_GRID = {
    "n_estimators":      [100, 300, 500],
    "max_depth":         [10, 20, 30, None],
    "min_samples_leaf":  [2, 5, 10],
    "class_weight":      ["balanced", "balanced_subsample"],
}

print(f"C grid: {len(C_GRID)} values")
print(f"MLP grid: {len(LR_GRID)} lr x {len(DROP_GRID)} dropout = {len(LR_GRID)*len(DROP_GRID)} combos")
print(f"RF grid: {np.prod([len(v) for v in RF_PARAM_GRID.values()])} combos")

In [ ]:
# ── Shared evaluation helper ──
def evaluate_model(model, test_sets, threshold=0.5):
    """Evaluate model on each test set. Returns dict of {name: {BalAcc, AUC}}."""
    results = {}
    for name, (X_t, y_t) in test_sets.items():
        if hasattr(model, 'predict_proba'):
            proba = model.predict_proba(X_t)[:, 1]
        else:
            proba = model.predict(X_t).astype(float)
        preds = (proba >= threshold).astype(int)
        results[name] = {
            "BalAcc": balanced_accuracy_score(y_t, preds),
            "AUC": roc_auc_score(y_t, proba),
        }
    return results

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# LOGISTIC REGRESSION
# ═══════════════════════════════════════════════════════════════════════════

def train_eval_lr(X_train, y_train, test_sets, label="LR"):
    """GridSearchCV over C + CV threshold tuning on X_train only."""
    print(f"\n{'='*60}")
    print(f"  {label}  |  train={X_train.shape[0]} samples")
    print(f"{'='*60}")

    grid = GridSearchCV(
        LogisticRegression(penalty="l2", solver="lbfgs", class_weight="balanced",
                           max_iter=5000, random_state=SEED),
        param_grid={"C": C_GRID}, cv=3, scoring="balanced_accuracy", n_jobs=-1)
    grid.fit(X_train, y_train)
    best_c = grid.best_params_["C"]
    print(f"  Best C = {best_c:.2e}")

    # CV threshold tuning
    cv_proba = cross_val_predict(
        LogisticRegression(C=best_c, penalty="l2", solver="lbfgs",
                           class_weight="balanced", max_iter=5000, random_state=SEED),
        X_train, y_train, cv=3, method="predict_proba", n_jobs=-1)[:, 1]

    best_t = THRESHOLDS[np.argmax(
        [balanced_accuracy_score(y_train, cv_proba >= t) for t in THRESHOLDS])]
    print(f"  CV threshold = {best_t:.3f}")

    # Retrain on full train
    lr = LogisticRegression(C=best_c, penalty="l2", solver="lbfgs",
                            class_weight="balanced", max_iter=5000, random_state=SEED)
    lr.fit(X_train, y_train)

    results = evaluate_model(lr, test_sets, threshold=best_t)
    for site, m in results.items():
        print(f"    {site:12s}  BalAcc={m['BalAcc']:.4f}  AUC={m['AUC']:.4f}")
    return results

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# MLP  (6x6 grid lr x dropout, threshold tuned on sub-val)
# ═══════════════════════════════════════════════════════════════════════════

def train_eval_mlp(X_train, y_train, test_sets, label="MLP"):
    print(f"\n{'='*60}")
    print(f"  {label}  |  train={X_train.shape[0]} samples")
    print(f"{'='*60}")

    # Split train into subtrain (85%) / subval (15%) for tuning
    X_st, X_sv, y_st, y_sv = train_test_split(
        X_train, y_train, test_size=0.15, stratify=y_train, random_state=SEED_TUNE)

    best_ba = -1.0
    best_lr, best_dh, best_t = None, None, 0.5

    for lr_val in LR_GRID:
        for d in DROP_GRID:
            dh, dl = d, d / 2.0
            mlp = MaldiMLPClassifier(
                hidden_dim=512, head_dims=(256, 128), use_attention=False,
                dropout_high=dh, dropout_low=dl, weight_decay=1e-3,
                learning_rate=lr_val, batch_size=64, epochs=50,
                early_stopping_patience=10, val_fraction=0.15,
                input_transform="none", tune_threshold=False, random_state=SEED)
            mlp.fit(X_st, y_st)
            proba = mlp.predict_proba(X_sv)[:, 1]
            for t in THRESHOLDS:
                ba = balanced_accuracy_score(y_sv, proba >= t)
                if ba > best_ba:
                    best_ba = ba
                    best_lr, best_dh, best_t = lr_val, dh, t
        print(f"  lr={lr_val:.1e}  best drop={best_dh:.1f}  best BA={best_ba:.4f}")

    print(f"\n  Best: lr={best_lr:.1e}  dropout={best_dh:.1f}  threshold={best_t:.3f}")

    # Retrain on full training set with warmup + early stopping
    mlp_final = MaldiMLPClassifier(
        hidden_dim=512, head_dims=(256, 128), use_attention=False,
        dropout_high=best_dh, dropout_low=best_dh / 2.0, weight_decay=1e-4,
        learning_rate=best_lr, batch_size=64, epochs=100,
        early_stopping_patience=15, warmup_epochs=10,
        val_fraction=0.15,
        input_transform="none", tune_threshold=False, random_state=SEED)
    print("  Retraining on full training set...")
    mlp_final.fit(X_train, y_train)

    results = evaluate_model(mlp_final, test_sets, threshold=best_t)
    for site, m in results.items():
        print(f"    {site:12s}  BalAcc={m['BalAcc']:.4f}  AUC={m['AUC']:.4f}")
    return results

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# RANDOM FOREST
# ═══════════════════════════════════════════════════════════════════════════

def train_eval_rf(X_train, y_train, test_sets, label="RF"):
    print(f"\n{'='*60}")
    print(f"  {label}  |  train={X_train.shape[0]} samples")
    print(f"{'='*60}")

    grid = GridSearchCV(
        RandomForestClassifier(oob_score=True, random_state=SEED, n_jobs=-1),
        param_grid=RF_PARAM_GRID, cv=3, scoring="balanced_accuracy",
        n_jobs=-1, verbose=0)
    grid.fit(X_train, y_train)
    rf = grid.best_estimator_
    print(f"  Best params: {grid.best_params_}")

    # CV threshold tuning
    cv_proba = cross_val_predict(
        RandomForestClassifier(**grid.best_params_, oob_score=True,
                               random_state=SEED, n_jobs=-1),
        X_train, y_train, cv=3, method="predict_proba", n_jobs=-1)[:, 1]

    best_t = THRESHOLDS[np.argmax(
        [balanced_accuracy_score(y_train, cv_proba >= t) for t in THRESHOLDS])]
    print(f"  CV threshold = {best_t:.3f}")

    results = evaluate_model(rf, test_sets, threshold=best_t)
    for site, m in results.items():
        print(f"    {site:12s}  BalAcc={m['BalAcc']:.4f}  AUC={m['AUC']:.4f}")
    return results

---
## Cross-Site  (train on A, test on A-val / B / C / D / B+C+D)

In [ ]:
# ── Cross-site test sets ──
cross_test_sets = {}
cross_test_sets["A-val"] = (X_A_val_pp, y_A_val)
for site in "BCD":
    if site in test_pp:
        cross_test_sets[site] = test_pp[site]
if "B+C+D" in test_pp:
    cross_test_sets["B+C+D"] = test_pp["B+C+D"]

print("Cross-site test sets:")
for name, (X_t, y_t) in cross_test_sets.items():
    print(f"  {name:10s}: {len(X_t)} samples")

In [ ]:
# ── Cross-site LR ──
cs_lr_results = train_eval_lr(X_A_train_pp, y_A_train, cross_test_sets, label="LR (cross-site)")

In [ ]:
# ── Cross-site MLP ──
cs_mlp_results = train_eval_mlp(X_A_train_pp, y_A_train, cross_test_sets, label="MLP (cross-site)")

In [ ]:
# ── Cross-site RF ──
cs_rf_results = train_eval_rf(X_A_train_pp, y_A_train, cross_test_sets, label="RF (cross-site)")

---
## Aggregated  (train on pooled A+B+C+D, test on agg-val holdout)

In [ ]:
# ── Aggregated test sets ──
agg_test_sets = {"Aggregated": (X_agg_val_pp, y_agg_val)}
print(f"Aggregated test set: {len(X_agg_val_pp)} samples")

In [ ]:
# ── Aggregated LR ──
agg_lr_results = train_eval_lr(X_agg_train_pp, y_agg_train, agg_test_sets, label="LR (aggregated)")

In [ ]:
# ── Aggregated MLP ──
agg_mlp_results = train_eval_mlp(X_agg_train_pp, y_agg_train, agg_test_sets, label="MLP (aggregated)")

In [ ]:
# ── Aggregated RF ──
agg_rf_results = train_eval_rf(X_agg_train_pp, y_agg_train, agg_test_sets, label="RF (aggregated)")

---
## Results

In [ ]:
# ── Assemble results into DataFrames ──

# Cross-site results
cs_results = {"LR": cs_lr_results, "MLP": cs_mlp_results, "RF": cs_rf_results}
agg_results = {"LR": agg_lr_results, "MLP": agg_mlp_results, "RF": agg_rf_results}

# Determine site order
cs_sites = []
for s in ["A-val", "B", "C", "D", "B+C+D"]:
    if s in cross_test_sets:
        cs_sites.append(s)
all_sites = cs_sites + ["Aggregated"]
print(f"Sites: {all_sites}")

# Build Balanced Accuracy DataFrame
ba_rows = []
for method in ["LR", "MLP", "RF"]:
    row = {"Method": method}
    for site in cs_sites:
        row[site] = cs_results[method][site]["BalAcc"]
    row["Aggregated"] = agg_results[method]["Aggregated"]["BalAcc"]
    ba_rows.append(row)
df_ba = pd.DataFrame(ba_rows).set_index("Method")

# Build AUC DataFrame
auc_rows = []
for method in ["LR", "MLP", "RF"]:
    row = {"Method": method}
    for site in cs_sites:
        row[site] = cs_results[method][site]["AUC"]
    row["Aggregated"] = agg_results[method]["Aggregated"]["AUC"]
    auc_rows.append(row)
df_auc = pd.DataFrame(auc_rows).set_index("Method")

print("\nBalanced Accuracy:")
print(df_ba.to_string())
print("\nAUC-ROC:")
print(df_auc.to_string())

In [ ]:
# ── Dual heatmaps ──

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 4.5))

# Balanced Accuracy heatmap
sns.heatmap(df_ba, annot=True, fmt=".3f", cmap="RdYlGn",
            vmin=0.5, vmax=1.0, linewidths=1.0, linecolor="white",
            cbar_kws={"label": "Balanced Accuracy", "shrink": 0.8}, ax=ax1)
ax1.set_title("Balanced Accuracy", fontsize=13, fontweight="bold")
ax1.set_xlabel(""); ax1.set_ylabel("Method")

# AUC heatmap
sns.heatmap(df_auc, annot=True, fmt=".3f", cmap="RdYlGn",
            vmin=0.5, vmax=1.0, linewidths=1.0, linecolor="white",
            cbar_kws={"label": "AUC-ROC", "shrink": 0.8}, ax=ax2)
ax2.set_title("AUC-ROC", fontsize=13, fontweight="bold")
ax2.set_xlabel(""); ax2.set_ylabel("Method")

fig.suptitle("Ceftazidime x E. coli  --  LR / MLP / RF  --  Cross-Site + Aggregated",
             fontsize=14, fontweight="bold", y=1.02)

plt.tight_layout()
plt.savefig(OUT_DIR / "heatmaps_balacc_auc.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# ── Optional: grouped bar chart view ──

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(all_sites))
w = 0.25
colors = {"LR": "#1f77b4", "MLP": "#ff7f0e", "RF": "#2ca02c"}

for i, method in enumerate(["LR", "MLP", "RF"]):
    values = [cs_results[method][s]["BalAcc"] if s in cs_results[method]
              else agg_results[method][s]["BalAcc"]
              for s in all_sites]
    ax.bar(x + (i - 1) * w, values, w, label=method, color=colors[method],
           edgecolor="white", linewidth=0.5)

ax.set_xticks(x)
ax.set_xticklabels(all_sites, fontsize=10)
ax.set_ylabel("Balanced Accuracy"); ax.set_title("Ceftazidime x E. coli -- LR / MLP / RF")
ax.legend(fontsize=10); ax.set_ylim(0, 1)
ax.axhline(0.5, color="gray", ls="--", alpha=0.4)
ax.grid(True, ls='--', lw=0.5, color='gray', alpha=0.5)

plt.tight_layout()
plt.savefig(OUT_DIR / "barchart_balacc.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# ── Save numeric results as CSV ──
df_ba.to_csv(OUT_DIR / "results_balanced_accuracy.csv")
df_auc.to_csv(OUT_DIR / "results_auc_roc.csv")
print("\nResults saved to:", OUT_DIR.resolve())
for f in sorted(OUT_DIR.glob("*")):
    print(f"  {f.name}")

---
**Done.**